# Gradient Matching 与 Distribution Matching

本章我们继续深入数据集蒸馏，为你介绍原始数据集蒸馏的进阶方案，Gradient Matching 与 Distribution Matching。我们试图减少原始数据集蒸馏计算量巨大的缺点并且进一步提升性能。

# Gradient Matching

实际上我们经常做这样一件事，就是指出优化的最小单位。比如 Flow Matching 指出优化模型输出的边缘矢量场最小单位是优化模型输出的条件矢量场，前者的 Ground-Truth 是几乎无法估计的，后者却可以被简单地写出。

Gradient Matching 的思想非常类似。我们发现原始数据集蒸馏的优化太过复杂，那么能否指出一个更小的优化单位？实际上这个最小单位完全存在，那就是梯度。

推荐你读 https://arxiv.org/abs/2006.05929 Dataset Condensation with Gradient Matching 这是 Gradient Matching 原文，同样是一篇开山之作。

我们详细说说。

## 基本逻辑

原始数据集蒸馏目标可以视为
$$\min_{\tilde{x}}
\ell\bigl(x,\theta_K(\tilde{x})\bigr)$$
我们希望蒸馏数据集可以促使模型优化时产生正确的梯度，并且根据该梯度优化之后的结果可以在真实数据集上表现良好。

那么为什么不直接对齐梯度本身？

我们换一个优化对象
$$\min_{\tilde{x}}
D\left(
\nabla_\theta\ell(x,\theta),
\nabla_\theta\ell(\tilde{x},\theta)
\right)$$
其中 $D(\cdot,\cdot)$ 是一个度量。

这里的思想非常简单。如果真实数据和合成数据在许多模型状态上都给出相近的下降方向，那么在合成数据上训练，应该近似在真实数据上训练。

我们开始正式地叙述。首先给定数据集 $\mathcal{T} = \{(x_i, y_i)\}$ 和网络 $\phi_\theta$，我们可以写出最优参数
$$\theta^\mathcal{T} = \arg\min_\theta \mathcal{L}^\mathcal{T}(\theta) = \frac{1}{|\mathcal{T}|} \sum_{(x,y)\in\mathcal{T}} \ell(\phi_\theta(x), y)$$
现在我们有蒸馏数据集 $\mathcal{S} = \{(s_i, y_i)\}, |\mathcal{S}| \ll |\mathcal{T}|$，我们写出最优参数
$$\theta^\mathcal{S} = \arg\min_\theta \mathcal{L}^\mathcal{S}(\theta)$$
我们希望最优参数 $\theta^\mathcal{S}$ 在真实数据集上表现好，这意味着我们要求蒸馏数据集是
$$\mathcal{S}^* = \arg\min_\mathcal{S} \mathcal{L}^\mathcal{T}(\theta^\mathcal{S}(\mathcal{S}))$$
以上是我们上一章讲述的朴素数据集蒸馏的思想。这里涉及到一个双层优化问题，导致计算非常昂贵。

现在我们逐步开始提出进阶的想法。首先，我们提出 Parameter Matching，我们希望蒸馏数据集优化参数和原始数据集优化参数可以直接匹配，这意味着他们的输出是相似的。换言之，我们希望
$$\min_{\mathcal{S}}
\mathbb{E}_{\theta_0\sim P_{\theta_0}}
\left[
D\left(
\theta^{\mathcal{S}}(\theta_0),
\theta^{\mathcal{T}}(\theta_0)
\right)
\right]$$
这还是一个昂贵的双层优化问题。因此我们提出，$\theta^{\mathcal{S}}$ 可以不是最优参数，而是经过固定步数优化后的最优
$$\theta^{\mathcal{S}}(\mathcal{S})
=
\operatorname{opt\text{-}alg}_{\theta}
\left(
\mathcal{L}^{\mathcal{S}}(\theta),
\varsigma
\right)$$
其中 $\operatorname{opt\text{-}alg}$ 表示某个优化算法，$\varsigma$ 表示固定优化步数。

Parameter Matching 的思想是，可以先训练好 $\theta^{\mathcal{T}}(\theta_0)$，然后从初始蒸馏数据集出发优化固定步数得到 $\theta^{\mathcal{S}}(\theta_0)$，然后计算我们需要的损失并且优化。

然而，一个巨大的问题是，训练初期 $\theta^{\mathcal{S}}$ 可能还距离 $\theta^{\mathcal{T}}$ 非常遥远，这意味对于损失 $D\left(
\theta^{\mathcal{S}},
\theta^{\mathcal{T}}
\right)$ 的优化过程充斥着噪声与局部极小值，目标跨度太大。

另一个巨大的问题是，即使两个神经网络实现完全相同的函数，它们的参数也可能差异很大。例如隐藏层神经元可以置换，若交换两个隐藏神经元，同时相应交换下一层权重，网络函数不变，但参数向量不同，这意味这个优化目标没有那么合理。

我们希望否决 Parameter Matching 并且提出其进阶。我们原本希望
$$\theta^{\mathcal{S}}
\approx
\theta^{\mathcal{T}}$$
现在改为
$$\nabla_\theta
\mathcal{L}^{\mathcal{S}}(\theta)
\approx
\nabla_\theta
\mathcal{L}^{\mathcal{T}}(\theta)$$
这就是 Gradient Matching 的基本思想。

下面这张图简单展示了这里的思想，无论是 Parameter Matching 还是 Gradient Matching，实际上都遵从右边这一套流程。

<img src="./assets/GM.png" width="700" height="250">

但是我们先不说 GM 的完整算法。一个重要技巧是 Curriculum Gradient Matching，简而言之就是我们希望解决上述优化梯度过大问题。我们提出，不只让合成数据训练出的模型最终接近 $\theta^\mathcal{T}$，还要沿着训练轨迹每一步都尽量接近对应参数轨迹 $\theta^\mathcal{T}_t$。

当我们对 PM 应用这一思想，优化目标变为
$$\min_{\mathcal{S}} \mathbb{E}_{\theta_0\sim P_{\theta_0}} \Big[ \sum_{t=0}^{T-1} D(\theta_t^\mathcal{S}, \theta_t^\mathcal{T}) \Big]$$
其中 $T$ 是迭代总步数，$\theta_t^\mathcal{S}$ 是在合成数据上训练 $t$ 步得到的参数，$\theta_t^\mathcal{T}$ 是是在真实数据上训练 $t$ 步得到的参数。

当我们对 GM 应用这一思想，首先我们写出梯度下降更新过程
$$\theta_{t+1}^\mathcal{S} \gets \theta_t^\mathcal{S} - \eta_\theta \nabla_\theta \mathcal{L}^\mathcal{S}(\theta_t^\mathcal{S})$$
$$\theta_{t+1}^\mathcal{T} \gets \theta_t^\mathcal{T} - \eta_\theta \nabla_\theta \mathcal{L}^\mathcal{T}(\theta_t^\mathcal{T})$$
最终优化目标是
$$\min_\mathcal{S} \mathbb{E}_{\theta_0 \sim P_{\theta_0}} \Big[ \sum_{t=0}^{T-1} D(\nabla_\theta \mathcal{L}^\mathcal{S}(\theta_t), \nabla_\theta \mathcal{L}^\mathcal{T}(\theta_t)) \Big]$$

这就是大概的思想。接下来我们说说具体算法。

## 蒸馏算法

给定权重初始化 $\theta_0 \sim P_{\theta_0}$。

对于迭代步数 $t=0,...,T-1$，对于每个类别 $c$ 抽取真实数据集 mini-batch $B_c^\mathcal{T}$ 和蒸馏数据集 mini-batch $B_c^\mathcal{S}$。我们可以计算梯度
$$\nabla_\theta \mathcal{L}_c^\mathcal{T}(\theta_t),\quad
\nabla_\theta \mathcal{L}_c^\mathcal{S}(\theta_t)$$
现在更新蒸馏数据集
$$\mathcal{S}_c \gets \text{opt-alg}_\mathcal{S}\Big(D(\nabla_\theta \mathcal{L}_c^\mathcal{S}, \nabla_\theta \mathcal{L}_c^\mathcal{T}),\varsigma_\mathcal{S},\eta_\mathcal{S}\Big)$$
然后更新 $\theta_{t+1}$
$$\theta_{t+1}
\leftarrow
\operatorname{opt\text{-}alg}_{\theta}
\left(
\mathcal{L}^{\mathcal{S}}(\theta_t),\varsigma_\theta,\eta_\theta
\right)$$
进入下一轮迭代。

关于这里的优化算法 $\text{opt-alg}$，实际上有很多选择，比如 SGD。上述的更新中，$\varsigma_\theta$ 给出了更新步数并且 $\eta_\theta$ 给出了更新步长。我们详细写开。记
$$\theta_t^{(0)}
=
\theta_t$$
对于 $r=0,1,\ldots,\varsigma_\theta-1$，执行
$$\theta_t^{(r+1)}
=
\theta_t^{(r)}
-
\eta_\theta
\nabla_\theta
\mathcal{L}^{\mathcal S}
\left(
\theta_t^{(r)}
\right)$$
最终
$$\theta_{t+1}
=
\theta_t^{(\varsigma_\theta)}$$
所以更新到 $\theta_{t+1}$ 需要计算很多次梯度 $\nabla_{\theta} \mathcal{L}^{\mathcal{S}}(\theta_t^{(r)})$。

对于 $\mathcal{S}_c$ 的更新是类似的，我们根据给定的 $\varsigma_{\mathcal{S}}$ 和 $\eta_{\mathcal{S}}$ 做更新
$$\mathcal{S}_c^{(r+1)}
=
\mathcal{S}_c^{(r)}
-
\eta_{\mathcal{S}_c}
\nabla_{\mathcal{S}_c}
D(\nabla_\theta \mathcal{L}_c^\mathcal{S}, \nabla_\theta \mathcal{L}_c^\mathcal{T})$$ 

现在我们详细说说这个度量 $D(\cdot,\cdot)$。神经网络中具有很多各种形状的层，我们需要定义一个距离衡量层之间差异。

其实非常简单，对于某层梯度，我们按照输出维度分割并且展平，这样可以得到输出维度个数个一维向量。我们记我们需要比较的向量为 $A, B$，那么度量就是
$$D(A,B) = \sum_i \Big(1 - \frac{A_i \cdot B_i}{\|A_i\|\|B_i\|}\Big)$$

更详细的，对于全连接层，权重是 $W\in\mathbb{R}^{n_{\mathrm{out}}\times n_{\mathrm{in}}}$，梯度会保持形状 $\nabla_W\ell\in\mathbb{R}^{n_{\mathrm{out}}\times n_{\mathrm{in}}}$。
我们记第 $i$ 行是
$$\left(\nabla_W\ell\right)_{i,:}
\in
\mathbb{R}^{n_{\mathrm{in}}}$$
对于真实数据第 $i$ 行产生梯度记为 $B_i$，对于蒸馏数据集产生梯度记为 $A_i$。有
$$A_i,B_i\in\mathbb{R}^{n_{\mathrm{in}}}$$
所以最后度量就是
$$D(A,B)
=
\sum_{i=1}^{n_{\mathrm{out}}}
\left(
1-
\frac{A_i\cdot B_i}
{\|A_i\|\|B_i\|}
\right)$$

解释一下为什么要分行计算，原因是第 $i$ 个输出 $z_i$ 可以写成
$$z_i
=
\sum_{j=1}^{n_{\mathrm{in}}}
W_{ij}x_j+b_i$$
因此梯度 $\nabla_{W_{i,:}}\ell$ 对这一行损失负责。

所以在卷积神经网络和自注意力模块上也做相同的事情，就是关于输出维度展开，换言之展开的元素维度就是原维度除以输出维度。最后度量求和都是按照输出维度求和。

下面是完整的 GM 算法。

$$\begin{array}{l}
\hline
\textbf{Algorithm 1:} \text{ Dataset condensation with gradient matching} \\
\hline
\textbf{Input:} \text{ Training set } \mathcal{T} \\
\begin{aligned}
1: & \ \textbf{Required:} \text{ Randomly initialized set of synthetic samples } \mathcal{S} \text{ for } C \text{ classes, probability distribution over} \\
   & \ \text{randomly initialized weights } P_{\boldsymbol{\theta}_0}, \text{ deep neural network } \phi_{\boldsymbol{\theta}}, \text{ number of outer-loop steps } K, \text{ number of} \\
   & \ \text{inner-loop steps } T, \text{ number of steps for updating weights } \varsigma_{\boldsymbol{\theta}} \text{ and synthetic samples } \varsigma_{\mathcal{S}} \text{ in each inner-loop} \\
   & \ \text{step respectively, learning rates for updating weights } \eta_{\boldsymbol{\theta}} \text{ and synthetic samples } \eta_{\mathcal{S}}. \\
2: & \ \textbf{for } k = 0, \dots, K - 1 \textbf{ do} \\
3: & \ \quad \text{Initialize } \boldsymbol{\theta}_0 \sim P_{\boldsymbol{\theta}_0} \\
4: & \ \quad \textbf{for } t = 0, \dots, T - 1 \textbf{ do} \\
5: & \ \quad\quad \textbf{for } c = 0, \dots, C - 1 \textbf{ do} \\
6: & \ \quad\quad\quad \text{Sample a minibatch pair } B_c^{\mathcal{T}} \sim \mathcal{T} \text{ and } B_c^{\mathcal{S}} \sim \mathcal{S} \qquad \qquad \ \ \triangleright B_c^{\mathcal{T}} \text{ and } B_c^{\mathcal{S}} \text{ are of the same class } c. \\
7: & \ \quad\quad\quad \text{Compute } \mathcal{L}_c^{\mathcal{T}} = \frac{1}{|B_c^{\mathcal{T}}|} \sum_{(\boldsymbol{x}, y) \in B_c^{\mathcal{T}}} \ell(\phi_{\boldsymbol{\theta}_t}(\boldsymbol{x}), y) \text{ and } \mathcal{L}_c^{\mathcal{S}} = \frac{1}{|B_c^{\mathcal{S}}|} \sum_{(\boldsymbol{s}, y) \in B_c^{\mathcal{S}}} \ell(\phi_{\boldsymbol{\theta}_t}(\boldsymbol{s}), y) \\
8: & \ \quad\quad\quad \text{Update } \mathcal{S}_c \leftarrow \text{opt-alg}_{\mathcal{S}}\left(D(\nabla_{\boldsymbol{\theta}}\mathcal{L}_c^{\mathcal{S}}(\boldsymbol{\theta}_t), \nabla_{\boldsymbol{\theta}}\mathcal{L}_c^{\mathcal{T}}(\boldsymbol{\theta}_t)), \varsigma_{\mathcal{S}}, \eta_{\mathcal{S}}\right) \\
9: & \ \quad \quad \text{Update } \boldsymbol{\theta}_{t+1} \leftarrow \text{opt-alg}_{\boldsymbol{\theta}}(\mathcal{L}^{\mathcal{S}}(\boldsymbol{\theta}_t), \varsigma_{\boldsymbol{\theta}}, \eta_{\boldsymbol{\theta}}) \qquad \qquad \qquad \qquad \qquad \qquad \triangleright \text{Use the whole } \mathcal{S} \\
10:& \ \textbf{end for}
\end{aligned} \\
\textbf{Output:} \ \mathcal{S} \\
\hline
\end{array}$$

## 成果与思考

GM 的成果很好。对于 MNIST 这样的简单数据集，在每类 $1$ 张情况下可以训练模型达到 $91.7\%$ 正确率，每类 $50$ 张则可以达到 $98.8\%$ 正确率。对于 Fashion-MNIST 这样略复杂的数据集，在每类 $50$ 张情况下可以到达到 $82.3\%$ 正确率，这个结果大幅超越了 K-Center 和 Random 这样的朴素数据集蒸馏方法。

但是我们需要讨论一件事，那就是 GM 的计算量到底相较原始数据集蒸馏方法减轻了多少？实际上 GM 本质还是存在一个双层优化问题。但是我们必须指出的一个关键事实是，GM 的计算图长度远远小于原始数据集蒸馏方法。

关于数据集蒸馏算法，其中关键一步更新是，对于 $r=0,1,\ldots,\varsigma_\theta-1$，执行
$$\mathcal{S}_c^{(r+1)}
=
\mathcal{S}_c^{(r)}
-
\eta_{\mathcal{S}_c}
\nabla_{\mathcal{S}_c}
D(\nabla_\theta \mathcal{L}_c^\mathcal{S}, \nabla_\theta \mathcal{L}_c^\mathcal{T})$$ 

这样写可能还不够清晰。对于固定的 $\theta_t$，我们分别计算在真实数据集和合成数据集上两个梯度
$$g_t^{\mathcal T}
=
\nabla_{\theta}
\mathcal L^{\mathcal T}(\theta_t)$$
$$g_t^{\mathcal S}
=
\nabla_{\theta}
\mathcal L^{\mathcal S}(\theta_t;\mathcal S)$$
那么损失就是
$$\mathcal L_{\mathrm{GM},t}
=
D\left(
g_t^{\mathcal S},
g_t^{\mathcal T}
\right)$$
所以梯度就是
$$\nabla_{\mathcal S}
\mathcal L_{\mathrm{GM},t}
=
\frac{\partial}{\partial\mathcal S}
D\left(
\nabla_\theta
\mathcal L^{\mathcal S}(\theta_t;\mathcal S),
g_t^{\mathcal T}
\right)$$
在这里求微分时，我们将 $\theta_t$ 和 $g_t^{\mathcal T}$ 视为常量。这意味着计算图非常局部。

所以计算图可以写开
$$\mathcal S
\longrightarrow
\mathcal L^{\mathcal S}(\theta_t;\mathcal S)
\longrightarrow
g_t^{\mathcal S}
=
\nabla_\theta\mathcal L^{\mathcal S}(\theta_t;\mathcal S)
\longrightarrow
D(g_t^{\mathcal S},g_t^{\mathcal T})
\longrightarrow
\nabla_{\mathcal S}\mathcal L_{\mathrm{GM},t}$$
或者从参数视角看就是
$$\operatorname{sg}(\theta_t)
\longrightarrow
\mathcal L^{\mathcal S}
\longrightarrow
\nabla_\theta\mathcal L^{\mathcal S}
\longrightarrow
D
\longrightarrow
\mathcal S$$

这里的停训算子 $\operatorname{sg}$ 非常重要。从全局关系上来看，数值上存在 
$$\theta_t
=
\operatorname{opt\text{-}alg}_\theta
\left(
\theta_{t-1},
\mathcal L^{\mathcal S}
\right)$$
但是 GM 将这一步忽略了，否则就需要像原始数据集蒸馏方法中那样计算非常多的中间量占据大量显存。

对于这个计算图，我们仍然可以应用 HVP 技术减少计算负担，因为这里还是存在一个二阶微分的 Hessian 矩阵。

所以结论是，毫无疑问 GM 计算量会比原始数据集蒸馏少很多，并且显存负担也非常少，这是巨大的进步。这个优势甚至会根据内层优化步数增加而扩大，原因是内层优化步数增加，GM 计算量基本不增加，但原始数据集蒸馏需要穿透的反向传播链路会更长。

## GM 的总结

GM 提出了比原始数据集蒸馏更加优秀的方法。不过目前为止，无论是原始数据集蒸馏还是 GM 方法，我们都希望通过一个模型来展示整个数据集的梯度，这就意味着我们无可避免地训练一个模型，导致计算的负担。

我们来看 Distribution Matching。我们指出，可以直接观察蒸馏数据集和原始数据集分布的特征来做蒸馏。这是全新的观点，这种方法可以避免二阶导的产生，从根源上减少了计算负担。

推荐你读 https://arxiv.org/pdf/2110.04181 Dataset Condensation with Distribution Matching 这是 Distribution Matching 原文。

# Distribution Matching

Distribution Matching 不再考虑参数梯度，而是先用一个特征提取网络 $\psi_\theta$ 把图像映射到特征空间
$$x
\longmapsto
\psi_\theta(x)$$
要求真实数据集和合成数据集在该特征空间中的分布接近。

最简单地说，就是匹配每一类的平均特征
$$\frac{1}{|\mathcal T_c|}
\sum_{x\in\mathcal T_c}
\psi_\theta(x)
\approx
\frac{1}{|\mathcal S_c|}
\sum_{s\in\mathcal S_c}
\psi_\theta(s)$$
所以目标函数也很自然
$$\mathcal L_{\mathrm{DM}}
=
\sum_{c=1}^{C}
\left\|
\frac{1}{|\mathcal T_c|}
\sum_{x\in\mathcal T_c}
\psi_\theta(x)
-
\frac{1}{|\mathcal S_c|}
\sum_{s\in\mathcal S_c}
\psi_\theta(s)
\right\|^2$$
其中 $C$ 是类别数量。

下面这张图展示了这种观点。我们希望合成数据集和真实数据集拥有相同的分布。
<img src="./assets/DM.png" width="450" height="210">

我们详细说说。

## 基本逻辑

我们拥有真实数据集 $\mathcal{T} = \{(x_i, y_i)\}$ 与合成数据集 $\mathcal{S} = \{(s_i, y_i)\}$，我们希望
$$\mathbb{E}_{x\sim P_D}[\ell(\phi_{\theta_{\mathcal{T}}}(x),y)] \approx \mathbb{E}_{x\sim P_D}[\ell(\phi_{\theta_{\mathcal{S}}}(x),y)]$$

对于数据 $x\in\mathbb{R}^d$，这个维度 $d$ 相当高，因此直接估计分布 $P_{\mathcal D}$ 是困难的。我们提出使用多个嵌入网络来估计分布
$$\psi_\vartheta:\mathbb{R}^d\to\mathbb{R}^{d'}$$
其中特征维度 $d'\ll d$，$\vartheta$ 是特征参数。

现在我们可以估计合成数据集和真实数据集之间差距，这种估计被称为 Maximum Mean Discrepancy
$$\text{MMD} = \sup_{\|\psi_\vartheta\|_{\mathcal H}\leq 1}
\left(
\mathbb E[\psi_\vartheta(\mathcal T)]
-
\mathbb E[\psi_\vartheta(\mathcal S)]
\right)$$
这里的 $\mathcal H$ 是 Reproducing Kernel Hilbert Space，$\psi_\vartheta$ 是允许使用的特征函数。这个损失实际上指，我们希望找出两个数据集之间差距最大的特征。如果所有特征差距都可以被控制，那么两个数据集就可以被视为拥有类似分布。

我必须解释一下什么是 Reproducing Kernel Hilbert Space，我们之后还会用到 RKHS 的诸多性质。首先 Hilbert 空间是完备的内积空间。对于一个 Hilbert 空间，我们要求，对于每个输入 $x\in\mathcal X$，存在核函数 $k(x,\cdot)\in\mathcal H$，使得对于任意 $f\in\mathcal H$，都有
$$f(x)
=
\langle f,k(x,\cdot)\rangle_{\mathcal H}$$
此处的核函数是指
$$k:\mathcal X\times\mathcal X\to\mathbb R$$
接收两个输入点 $x,x'$，输出一个相似度
$$ k(x,x')$$

请注意，根据 Moore–Aronszajn 定理，一个 RKHS 与一个核函数是一一对应的。这个性质告诉我们，对于多个核函数诱导的 RKHS 的讨论是不必要的，因为这种 RKHS 根本不存在。

比较抽象。我们给出一个例子，高斯核函数诱导的 RKHS。定义高斯核
$$k(x,x') = \exp\Big(-\frac{\|x-x'\|^2}{2\sigma^2}\Big)$$
而空间是，对于任意 $f \in \mathcal H$ 可以写成线性组合
$$f(\cdot) = \sum_{i=1}^n \alpha_i k(x_i, \cdot)$$
其中 $\alpha_i \in \mathbb R$，$\{x_i\}$ 是数据点。

内积法则是，对于 $f = \sum_{i=1}^n \alpha_i k(x_i,\cdot)$ 和 $g = \sum_{j=1}^m \beta_j k(y_j,\cdot)$
$$\langle f, g \rangle_{\mathcal H} = \sum_{i=1}^n \sum_{j=1}^m \alpha_i \beta_j k(x_i, y_j)$$

所以 RKHS 性质体现在
$$f(x) = \left\langle f, k(x,\cdot) \right\rangle_{\mathcal H} = \sum_{i=1}^n \alpha_i k(x_i, x)$$

但是此处的 Distribution Matching 为什么要求了 RKHS？实际上是因为我们依赖 RKHS 性质对特征函数做变化。对于
$$\operatorname{MMD}(P,Q)
=
\sup_{\|f\|_{\mathcal H}\leq 1}
\left(
\mathbb E_{x\sim P}[f(x)]
-
\mathbb E_{y\sim Q}[f(y)]
\right)$$
我们可以利用核函数
$$\mu_P
=
\mathbb E_{x\sim P}[k(x,\cdot)]$$
以及
$$\mu_Q
=
\mathbb E_{y\sim Q}[k(y,\cdot)]$$
可以得到
$$\operatorname{MMD}(P,Q)
=
\sup_{\|f\|_{\mathcal H}\leq 1}
\left\langle
f,\mu_P-\mu_Q
\right\rangle_{\mathcal H}$$
由 Cauchy–Schwarz 不等式
$$\operatorname{MMD}(P,Q)
=
\|\mu_P-\mu_Q\|_{\mathcal H}$$

利用上述的变换，我们可以给出工程上的写法。更多的，我们在工程中只可以使用有限个特征网络与有限个样本估计，也就是说
$$\text{MMD} = \mathbb E_{\vartheta\sim P_\vartheta}
\left\|
\frac{1}{|\mathcal T|}
\sum_{i=1}^{|\mathcal T|}
\psi_\vartheta(x_i)
-
\frac{1}{|\mathcal S|}
\sum_{j=1}^{|\mathcal S|}
\psi_\vartheta(s_j)
\right\|^2$$
我们记真实特征平均是
$$\mu_{\mathcal T}^{\vartheta}
=
\frac{1}{|\mathcal T|}
\sum_{i=1}^{|\mathcal T|}
\psi_\vartheta(x_i)$$
合成特征平均是
$$\mu_{\mathcal S}^{\vartheta}
=
\frac{1}{|\mathcal S|}
\sum_{j=1}^{|\mathcal S|}
\psi_\vartheta(s_j)$$
那么目标就是
$$\mathbb E_{\vartheta\sim P_\vartheta}
\left\|
\mu_{\mathcal T}^{\vartheta}
-
\mu_{\mathcal S}^{\vartheta}
\right\|^2$$

不过需要注意，这里的 $\mu_{\mathcal T}^{\vartheta}$ 和 $\mu_{\mathcal S}^{\vartheta}$ 并不是严格的核函数，我们使用多个特征神经网络试图近似核函数的思想。

最后的，这里还有一个技巧是 Siamese augmentation。我们记算子
$$\mathcal A(\cdot,\omega)$$
其中 $\omega$ 是随机增强参数。这个算子的意思是对于图像做某种程度变换，比如旋转和剪切，程度由参数 $\omega$ 决定。这是为了进一步提取深层特征，保证数据集不是仅仅从某个角度看上去相似。

所以最终优化目标是
$$\min_{\mathcal S}
\mathbb E_{
\substack{
\vartheta\sim P_\vartheta\\
\omega\sim\Omega
}
}
\left\|
\frac{1}{|\mathcal T|}
\sum_{i=1}^{|\mathcal T|}
\psi_\vartheta\bigl(\mathcal A(x_i,\omega)\bigr)
-
\frac{1}{|\mathcal S|}
\sum_{j=1}^{|\mathcal S|}
\psi_\vartheta\bigl(\mathcal A(s_j,\omega)\bigr)
\right\|^2$$
其中 $\Omega$ 是数据增强参数分布。

请注意优化这个损失的计算量。计算图是
$$\mathcal S
\longrightarrow
\mathcal A(\mathcal S,\omega)
\longrightarrow
\psi_\vartheta(\mathcal A(\mathcal S,\omega))
\longrightarrow
\mathcal L_{\mathrm{DM}}$$
这意味着梯度下降仅仅需要一阶梯度
$$\mathcal S
\leftarrow
\mathcal S
-
\eta
\nabla_{\mathcal S}\mathcal L_{\mathrm{DM}}$$
我们完全避免了二阶微分的巨大计算量。

现在我们详细说说这个蒸馏算法。

## 蒸馏算法

对于迭代次数 $k=0,...,K-1$，我们做以下事。

首先采样特征函数参数 $\theta \sim P_\theta$。对于每个类别 $c$ 采样 mini-batch $B_c^\mathcal{T} \sim \mathcal{T}$ 和 $B_c^\mathcal{S} \sim \mathcal{S}$，以及数据增强参数 $\omega_c \sim \Omega$。

对于每个类别，按照我们上述的损失公式进行计算后求和。

反向传播梯度下降更新蒸馏数据集 $\mathcal{S}$。

以下是完整算法。

$$\begin{array}{l}
\hline
\textbf{Algorithm 1:} \text{ Dataset condensation with distribution matching} \\
\hline
\textbf{Input:} \text{ Training set } \mathcal{T} \\
\begin{aligned}
1: & \ \textbf{Required:} \text{ Randomly initialized set of synthetic samples } \mathcal{S} \text{ for } C \text{ classes, deep neural network } \psi_{\boldsymbol{\vartheta}} \text{ parameterized} \\
   & \ \text{with } \boldsymbol{\vartheta}, \text{ probability distribution over parameters } P_{\boldsymbol{\vartheta}}, \text{ differentiable augmentation } \mathcal{A}_{\omega} \text{ parameterized with } \omega, \\
   & \ \text{augmentation parameter distribution } \Omega, \text{ training iterations } K, \text{ learning rate } \eta. \\
2: & \ \textbf{for } k = 0, \dots, K - 1 \textbf{ do} \\
3: & \ \quad \text{Sample } \boldsymbol{\vartheta} \sim P_{\boldsymbol{\vartheta}} \\
4: & \ \quad \text{Sample mini-batch pairs } B_c^{\mathcal{T}} \sim \mathcal{T} \text{ and } B_c^{\mathcal{S}} \sim \mathcal{S} \text{ and } \omega_c \sim \Omega \text{ for every class } c \\
5: & \ \quad \text{Compute } \mathcal{L} = \sum_{c=0}^{C-1} \left\| \frac{1}{|B_c^{\mathcal{T}}|} \sum_{(\boldsymbol{x}, y) \in B_c^{\mathcal{T}}} \psi_{\boldsymbol{\vartheta}}(\mathcal{A}_{\omega_c}(\boldsymbol{x})) - \frac{1}{|B_c^{\mathcal{S}}|} \sum_{(\boldsymbol{s}, y) \in B_c^{\mathcal{S}}} \psi_{\boldsymbol{\vartheta}}(\mathcal{A}_{\omega_c}(\boldsymbol{s})) \right\|^2 \\
6: & \ \quad \text{Update } \mathcal{S} \leftarrow \mathcal{S} - \eta \nabla_{\mathcal{S}}\mathcal{L} \\
7: & \ \textbf{end for}
\end{aligned} \\
\textbf{Output:} \ \mathcal{S} \\
\hline
\end{array}$$

我们讨论几件事。首先，为什么是从分布 $P_\theta$ 中随机挑选初始化特征函数权重而不是准备一些预训练权重？原因是作者发现两者效果类似，而后者明显计算成本更高。

其次的，我们指出，GM 其实是 DM 选取了一种特定的特征函数。原因是我们可以发现两者具有非常详细的形式。对预测不准确的样本，GM 会给更大的特征函数的权重，并且对不同网络和训练迭代，这些权重动态变化。所以 DM 实际上拥有比 GM 更弱的特征函数，但是换来了更低的计算成本。

还有一件事是，DM 还简化了调参过程。GM 和原始数据集蒸馏需要调整许多超参数和学习率，但是 DM 只需要调整一个蒸馏数据集学习率 $\eta$ 即可。

这里有一件事非常有趣：我们能否用生成式模型生成的图像作为蒸馏数据集？这个问题的答案是，至少朴素生成是不行的，原因是生成式模型追求图像真实感，这和训练分类器效率不完全相关。更多的，实验表明生成式模型生成的图像作为数据集甚至不一定比随机选择真实样本更有效。

# 总结

本章介绍了 GM 和 DM，我们已经逐渐步入主流的数据集蒸馏方案。

下一章为你带来 Matching Training Trajectories，这是另一种实用的思路。